In [1]:
import cv2
import torch
import socket
import numpy as np
from PIL import Image
import torch.nn as nn
import mediapipe as mp
import torchvision.transforms as transforms

In [3]:
# For webcam input:
video_path = 'D:\Ramana\Carrer\Projects\CV Gun Game\InputVid1.mp4'
cap = cv2.VideoCapture(video_path)

In [2]:
#Communication
sock = socket.socket(socket.AF_INET,socket.SOCK_DGRAM)
serverAddressPort = ("127.0.0.1", 5052)

In [3]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

In [4]:
import cv2
import torch
import socket
import numpy as np
from PIL import Image
import torch.nn as nn
import mediapipe as mp
import torchvision.transforms as transforms


# Communication
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
serverAddressPort = ("127.0.0.1", 5052)

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# Video Hand Detection
cap = cv2.VideoCapture(0)
with mp_hands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5, max_num_hands=2) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_frame)

        # Initialize directions for both hands
        right_direction = "None"
        left_direction = "None"

        if results.multi_hand_landmarks:
            for i, hand_landmarks in enumerate(results.multi_hand_landmarks):
                handedness = results.multi_handedness[i].classification[0].label
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                
                thumb_tip = hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_TIP]
                thumb_ip = hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_IP]
                index_mcp = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_MCP]
                
                if handedness == "Right":  # Right hand logic
                    thumb_up = thumb_tip.y < thumb_ip.y and thumb_tip.y < index_mcp.y
                    thumb_down = thumb_tip.y > thumb_ip.y and thumb_tip.y > index_mcp.y
                    thumb_right = thumb_tip.x > thumb_ip.x
                    thumb_left = thumb_tip.x < thumb_ip.x
                    
                    if thumb_up:
                        right_direction = "Forward"
                    elif thumb_down:
                        right_direction = "Backward"
                    elif thumb_right:
                        right_direction = "Right"
                    elif thumb_left:
                        right_direction = "Left"
                    else:
                        right_direction = "None"
                
                elif handedness == "Left":  # Left hand logic (only left and right)
                    thumb_right = thumb_tip.x > thumb_ip.x
                    thumb_left = thumb_tip.x < thumb_ip.x
                    
                    if thumb_right:
                        left_direction = "Right"
                    elif thumb_left:
                        left_direction = "Left"
                    else:
                        left_direction = "None"

        # Prepare and send UDP data as an array
        udp_data = f"[{right_direction},{left_direction}]"
        sock.sendto(str.encode(udp_data), serverAddressPort)
        
        # Display directions on the frame
        cv2.rectangle(frame, (10, 10), (300, 90), (0, 0, 0), -1)
        cv2.putText(frame, f"Right: {right_direction}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(frame, f"Left: {left_direction}", (20, 70), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        
        cv2.imshow('Dual Hand Tracker', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()